# Building StudyBot with Google ADK

I started this project because I was curious how hard it would be to build something useful with Google's Agent Development Kit. Turns out, not that hard — once you get past the initial config headaches.

This notebook walks through the whole thing: what ADK actually is, why the tool pattern makes sense, and how to wire it all together into an agent that can actually help you study. I wrote it so I could come back to it later and not have to reverse-engineer my own code.

By the end you'll have a working StudyBot that can explain topics, quiz you, and make a study plan. We'll also run it in the ADK web UI which is genuinely pretty cool.

**What you need before starting:**
- Python 3.10 or newer
- A Google AI Studio API key (free at [aistudio.google.com](https://aistudio.google.com/apikey))
- A terminal and a basic comfort level with pip

---

## A quick note on how ADK works

Google ADK is a framework for building agents — basically, LLMs that can call functions you define. The core idea is simple:

1. You write Python functions ("tools") that do specific things
2. You give those functions docstrings so the model knows what they do and when to use them
3. You create an `Agent` and pass it those tools
4. When a user sends a message, the agent decides on its own whether to call a tool and which one

The part that took me a minute to wrap my head around: the tools don't have to actually *do* the work themselves. In our case they return prompt strings, and the LLM does the generation. So the tool is basically telling the model "here's the format I want" and the model fills it in. It's a simple pattern but it works really well for structured outputs.

If you want a more complex agent where tools call external APIs or run code, the same structure applies — you just put real logic in the functions instead.

## Getting the environment set up

If you cloned the repo, you should already have the files. Let's make sure dependencies are installed and then we'll look at what's actually in each file.

Run this once to install everything:

In [ ]:
# Install the ADK package — this is the only real dependency
# If you're in a virtual environment (you should be), this is safe to run
%pip install -q google-adk==1.28.1

Now let's set the API key. The agent code also loads it from a `.env` file in the `studybot/` directory, but for the notebook it's easier to just set it directly:

In [ ]:
import os

# If you have a .env file, you can load it here instead
# from dotenv import load_dotenv; load_dotenv('studybot/.env')

# Or just set it directly for testing (don't commit this!)
# os.environ['GOOGLE_API_KEY'] = 'your-key-here'

if not os.getenv('GOOGLE_API_KEY'):
    print("⚠️  GOOGLE_API_KEY isn't set. The agent cells below won't work.")
    print("   Set it above or create studybot/.env with your key.")
else:
    print("✓ API key found")

---

## The tools

Let's look at `studybot/tools.py`. This is the most interesting file to understand because it shows exactly how ADK tools work.

The three tools are:
- `explain_topic` — takes a topic and a level, returns a prompt asking for an explanation
- `quiz_student` — takes a topic, returns a prompt that makes the model generate quiz questions
- `study_planner` — takes a subject and number of days, returns a prompt for a structured study plan

The docstrings matter a lot here. ADK uses them to decide when to call each tool.

In [ ]:
# Let's look at what the tools actually return before we hook them into an agent

import sys
sys.path.insert(0, '.')  # make sure we can import from the repo root

from studybot.tools import explain_topic, quiz_student, study_planner

# explain_topic just returns a formatted string — the LLM handles the actual explanation
print(explain_topic('recursion', level='beginner'))

In [ ]:
# And the quiz tool
print(quiz_student('Python lists'))

In [ ]:
# Study planner
print(study_planner('Data Structures', days_until_exam=5))

These are just strings, which might feel underwhelming at first. But when these return values go back to the Gemini model, it uses them as structured prompts to generate actual, useful output. The tools are essentially giving the model a template to fill in.

Here's the actual code (from `studybot/tools.py`) so you can see what's happening:

In [ ]:
# You can read the source directly
with open('studybot/tools.py', 'r') as f:
    print(f.read())

---

## The agent

`studybot/agent.py` is where everything comes together. It does a few things:

1. Loads the API key from a `.env` file if one exists (so you don't have to set env vars manually every time)
2. Creates the `root_agent` — an `Agent` instance with a name, description, instruction, and the list of tools

The `instruction` field is basically the system prompt. It tells the model its personality and how to behave. I kept it short here — just enough to make it actually use the tools instead of trying to answer everything from memory.

In [ ]:
with open('studybot/agent.py', 'r') as f:
    print(f.read())

One thing worth noting: the `model` is configurable via the `STUDYBOT_MODEL` environment variable. Defaults to `gemini-2.5-flash-lite` which is fast and cheap enough for testing. You can swap it for `gemini-2.5-pro` if you want better quality responses.

---

## Talking to the agent programmatically

Before we fire up the web UI, let's run the agent in code so you can see the full flow. ADK uses an async runner pattern — a bit verbose but it makes sense once you see it a couple of times.

In [ ]:
import asyncio
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

from studybot.agent import root_agent


async def ask_studybot(question: str) -> str:
    """Send a single message to StudyBot and return the response text."""
    app_name = "notebook-demo"
    user_id = "notebook-user"

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name=app_name, user_id=user_id)

    runner = Runner(
        app_name=app_name,
        agent=root_agent,
        session_service=session_service,
    )

    message = types.Content(
        role="user",
        parts=[types.Part(text=question)],
    )

    chunks = []
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=message,
    ):
        if event.content and event.content.parts:
            for part in event.content.parts:
                text = getattr(part, 'text', None)
                if text:
                    chunks.append(text)

    return ''.join(chunks).strip()


# Test it out — this will call the real API
response = await ask_studybot("Explain Python decorators at a beginner level.")
print(response)

In [ ]:
# Try the study planner
response = await ask_studybot("I have an exam on algorithms in 3 days. Make me a study plan.")
print(response)

In [ ]:
# And the quiz
response = await ask_studybot("Quiz me on Python dictionaries.")
print(response)

---

## Running the web UI

The programmatic approach is useful for testing, but the ADK web UI is much nicer for actually using the agent. It gives you a chat interface and shows you the tool calls happening in real time.

From the repo root (in a terminal, not the notebook — the server will block the process):

```bash
adk web . --host 127.0.0.1 --port 8000 --allow_origins "regex:.*"
```

Then open [http://127.0.0.1:8000/dev-ui](http://127.0.0.1:8000/dev-ui). You should see StudyBot in the agent list. Select it and start chatting.

The `--allow_origins "regex:.*"` flag is there because I was developing in a remote container and kept hitting CORS errors. If you're running this locally on your own machine you might not need it, but it doesn't hurt.

**If responses seem to cut off or stall:** There's a streaming toggle in the top-right corner of the Dev UI. Try turning it off and resending your message — sometimes the streaming mode is flaky depending on your setup.

---

## Smoke test

There's a smoke test script in `scripts/smoke_test.py` that sends a simple message and checks that the agent responds. Useful to run after any config changes to confirm things are still working.

```bash
python scripts/smoke_test.py
```

Or from the notebook:

In [ ]:
# Quick inline smoke test — same logic as scripts/smoke_test.py
response = await ask_studybot("Explain Python functions in two sentences.")
if response:
    print("✓ Agent responded")
    print(response[:300])
else:
    print("✗ No response — check your API key and retry")

---

## Where to go from here

This is a pretty minimal agent, but the structure scales well. A few things I'd add if I was taking this further:

**Add memory between sessions** — right now each conversation starts fresh. ADK supports persistent session storage so you could track what topics a student has already covered.

**Connect to real data** — the tools currently just return prompts. You could swap them for actual logic: pull notes from a file, check a student's progress in a database, generate quizzes from a real question bank.

**Add more tools** — flashcard generation, concept mapping, recommended YouTube links based on topic — all of this fits naturally into the same tool pattern.

**Swap the model** — set `STUDYBOT_MODEL=gemini-2.5-pro` in your `.env` for noticeably better explanations, especially on complex topics. Flash-lite is fine for prototyping but the quality difference is real.

The full ADK docs are at [google.github.io/adk-docs](https://google.github.io/adk-docs) — worth a read once you've got the basics down.